# Treinamento de Modelos — Credit Risk Intelligence Platform

Este notebook treina, compara e avalia múltiplos modelos de classificação para previsão de inadimplência (credit risk default). Inclui Dummy Classifier (baseline), Logistic Regression, Random Forest, XGBoost e LightGBM.

**Dataset**: `credit_risk.gold.ml_train` (307.511 registros, 229 features)
**Objetivo**: Identificar modelos com melhor desempenho para previsão de inadimplência.
**Importante**: NÃO registra no MLflow, não publica modelos, não cria APIs, não faz deploy.

In [0]:
# ============================================================
# INSTALAÇÃO DE BIBLIOTECAS ADICIONAIS (XGBoost E LightGBM)
# Caso não estejam disponíveis, o notebook continuará com WARNING
# ============================================================
%pip install xgboost lightgbm -q

In [0]:
# ============================================================
# SEÇÃO 0 — IMPORTS E CONFIGURAÇÃO INICIAL
# ============================================================
import warnings
import time
import uuid
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Scikit-Learn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, recall_score,
    precision_score, f1_score, accuracy_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    classification_report
)

# Tentar importar XGBoost e LightGBM
xgb_available = False
lgb_available = False

try:
    from xgboost import XGBClassifier
    xgb_available = True
    print("✅ XGBoost disponível")
except ImportError:
    print("⚠️ WARNING: XGBoost não disponível — modelo será pulado")

try:
    from lightgbm import LGBMClassifier
    lgb_available = True
    print("✅ LightGBM disponível")
except ImportError:
    print("⚠️ WARNING: LightGBM não disponível — modelo será pulado")

# Configurações globais
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# ID de execução para auditoria
EXECUTION_ID = str(uuid.uuid4())
START_TIME = time.time()
START_TIMESTAMP = datetime.now()

# Dicionários globais para armazenar resultados
all_results = []         # Lista de dicts com métricas de cada modelo
trained_models = {}      # Modelos treinados {nome: pipeline}
cv_results = {}           # Resultados de cross-validation {nome: {mean, std}}
feature_importances = {}  # Importâncias de features {nome: DataFrame}

print(f"\n{'='*60}")
print(f"TRAINING NOTEBOOK — CREDIT RISK INTELLIGENCE PLATFORM")
print(f"Execution ID: {EXECUTION_ID}")
print(f"Start Time: {START_TIMESTAMP}")
print(f"{'='*60}")

In [0]:
# ============================================================
# SEÇÃO 1 — CARREGAMENTO DOS DADOS
# ============================================================
print("=" * 60)
print("SEÇÃO 1 — Carregamento dos Dados")
print("=" * 60)

# Carregar tabelas do Unity Catalog
print("Carregando dados...")
df_train = spark.table("credit_risk.gold.ml_train").toPandas()
df_test = spark.table("credit_risk.gold.ml_test").toPandas()
df_metadata = spark.table("credit_risk.gold.ml_feature_metadata").toPandas()

# Validações básicas
print(f"\n--- Dimensões ---")
print(f"ml_train: {df_train.shape[0]:,} registros, {df_train.shape[1]} colunas")
print(f"ml_test:  {df_test.shape[0]:,} registros, {df_test.shape[1]} colunas")
print(f"ml_feature_metadata: {df_metadata.shape[0]} features documentadas")

# Verificar TARGET existe apenas no train
print(f"\n--- Validação TARGET ---")
target_in_train = 'TARGET' in df_train.columns
target_in_test = 'TARGET' in df_test.columns
print(f"TARGET em ml_train: {'✅' if target_in_train else '❌'}")
print(f"TARGET em ml_test:  {'❌ (esperado)' if not target_in_test else '⚠️ ERRO!'}")

# Distribuição do TARGET
print(f"\n--- Distribuição do TARGET (Train) ---")
target_dist = df_train['TARGET'].value_counts().sort_index()
target_pct = df_train['TARGET'].value_counts(normalize=True).sort_index() * 100
for val, cnt, pct in zip(target_dist.index, target_dist.values, target_pct.values):
    print(f"  TARGET={val}: {cnt:,} ({pct:.2f}%)")
minority_ratio = target_dist[0] / target_dist[1]
print(f"  Ratio majoritária/minoritária: {minority_ratio:.2f}:1")

# Verificar SK_ID_CURR
print(f"\n--- SK_ID_CURR ---")
print(f"SK_ID_CURR em ml_train: {'✅' if 'SK_ID_CURR' in df_train.columns else '❌'}")
print(f"SK_ID_CURR em ml_test:  {'✅' if 'SK_ID_CURR' in df_test.columns else '❌'}")
print(f"SK_ID_CURR NÃO será utilizado como feature (apenas identificador)")

# Tipos das colunas
print(f"\n--- Tipos de Dados (ml_train) ---")
dtype_counts = df_train.dtypes.value_counts()
for dtype, count in dtype_counts.items():
    print(f"  {dtype}: {count} colunas")

# Colunas categóricas (string)
categorical_cols_overview = df_train.select_dtypes(include=['object']).columns.tolist()
print(f"\n--- Colunas Categóricas ---")
print(f"Total: {len(categorical_cols_overview)} colunas string")
print(f"Colunas: {categorical_cols_overview}")

print("\n✅ Carregamento concluído com sucesso!")

In [0]:
# ============================================================
# SEÇÃO 2 — PREPARAÇÃO DOS DADOS
# ============================================================
print("=" * 60)
print("SEÇÃO 2 — Preparação dos Dados")
print("=" * 60)

# Separar features (X) e target (y)
# Excluir SK_ID_CURR (identificador) e TARGET (variável resposta)
exclude_cols = ['SK_ID_CURR', 'TARGET']
feature_cols = [c for c in df_train.columns if c not in exclude_cols]

X = df_train[feature_cols].copy()
y = df_train['TARGET'].copy()

print(f"X (features): {X.shape}")
print(f"y (target): {y.shape}")
print(f"Colunas excluídas: {exclude_cols}")

# Identificar colunas numéricas e categóricas
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
print(f"\nFeatures numéricas: {len(numeric_features)}")
print(f"Features categóricas: {len(categorical_features)}")

# --- Validação de valores nulos ---
print(f"\n--- Valores Nulos ---")
null_stats = X.isnull().sum()
null_pct = (X.isnull().sum() / len(X) * 100).round(2)
null_summary = pd.DataFrame({
    'coluna': null_stats.index,
    'nulos': null_stats.values,
    'percentual': null_pct.values
}).sort_values('percentual', ascending=False)

cols_with_nulls = null_summary[null_summary['nulos'] > 0]
print(f"Colunas com valores nulos: {len(cols_with_nulls)} de {len(X.columns)}")
if len(cols_with_nulls) > 0:
    print("Top 10 colunas com mais nulos:")
    print(cols_with_nulls.head(10).to_string(index=False))

# --- Validação de tipos incompatíveis ---
print(f"\n--- Tipos Incompatíveis ---")
incompatible_types = []
for col in X.columns:
    if X[col].dtype == 'object':
        unique_types = set(type(v).__name__ for v in X[col].dropna().unique()[:100])
        if len(unique_types) > 1:
            incompatible_types.append((col, unique_types))
if incompatible_types:
    print(f"⚠️ Colunas com tipos mistos: {incompatible_types}")
else:
    print("✅ Nenhum tipo incompatível detectado")

# --- Colunas constantes (documentar, NÃO remover) ---
print(f"\n--- Colunas Constantes ---")
constant_cols = []
for col in X.columns:
    if X[col].nunique() <= 1:
        constant_cols.append(col)

if constant_cols:
    print(f"⚠️ {len(constant_cols)} colunas constantes detectadas (documentadas, NÃO removidas):")
    for c in constant_cols:
        print(f"  - {c} (valor único: {X[c].unique()[0]})")
else:
    print("✅ Nenhuma coluna constante detectada")

constant_cols_documented = constant_cols.copy()

# --- Preprocessor para Pipeline sklearn ---
# Numérico: imputar mediana + escalonar
# Categórico: imputar mais frequente + one-hot encoding
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

print(f"\n--- Preprocessor Configurado ---")
print(f"  Numérico: SimpleImputer(median) + StandardScaler ({len(numeric_features)} cols)")
print(f"  Categórico: SimpleImputer(most_frequent) + OneHotEncoder ({len(categorical_features)} cols)")
print("\n✅ Preparação concluída!")

In [0]:
# ============================================================
# SEÇÃO 3 — DIVISÃO DE VALIDAÇÃO (STRATIFIED SPLIT)
# ============================================================
print("=" * 60)
print("SEÇÃO 3 — Divisão de Validação (Stratified Split 80/20)")
print("=" * 60)

# Stratified split 80/20 para preservar distribuição do TARGET
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(f"Train interno:  {X_tr.shape[0]:,} registros ({X_tr.shape[0]/len(X)*100:.1f}%)")
print(f"Validation:     {X_val.shape[0]:,} registros ({X_val.shape[0]/len(X)*100:.1f}%)")

# Registrar distribuição das classes
print(f"\n--- Distribuição das Classes ---")
print(f"Train interno:")
for val, cnt in y_tr.value_counts().sort_index().items():
    pct = cnt / len(y_tr) * 100
    print(f"  TARGET={val}: {cnt:,} ({pct:.2f}%)")

print(f"Validation:")
for val, cnt in y_val.value_counts().sort_index().items():
    pct = cnt / len(y_val) * 100
    print(f"  TARGET={val}: {cnt:,} ({pct:.2f}%)")

# Calcular scale_pos_weight para modelos boosting
n_pos = int(y_tr.sum())
n_neg = int(len(y_tr) - n_pos)
scale_pos_weight = n_neg / n_pos
print(f"\nscale_pos_weight (para XGBoost/LightGBM): {scale_pos_weight:.4f}")
print(f"  Positivos (default): {n_pos:,} | Negativos (no default): {n_neg:,}")

print("\n✅ Divisão estratificada concluída!")

In [0]:
# ============================================================
# SEÇÃO 4 — BASELINE (DUMMY CLASSIFIER)
# ============================================================
print("=" * 60)
print("SEÇÃO 4 — Baseline (Dummy Classifier)")
print("=" * 60)

# Baseline 1: Sempre classe majoritária
dummy_majority = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DummyClassifier(strategy='most_frequent', random_state=42))
])

print("Treinando Dummy (most_frequent)...")
dummy_majority.fit(X_tr, y_tr)
y_pred_dm = dummy_majority.predict(X_val)
y_proba_dm = dummy_majority.predict_proba(X_val)[:, 1]

# Métricas
roc_dm = roc_auc_score(y_val, y_proba_dm)
pr_dm = average_precision_score(y_val, y_proba_dm)
recall_dm = recall_score(y_val, y_pred_dm)
precision_dm = precision_score(y_val, y_pred_dm, zero_division=0)
f1_dm = f1_score(y_val, y_pred_dm, zero_division=0)
acc_dm = accuracy_score(y_val, y_pred_dm)

print(f"\nDummy (most_frequent):")
print(f"  ROC-AUC: {roc_dm:.4f} | PR-AUC: {pr_dm:.4f}")
print(f"  Recall: {recall_dm:.4f} | Precision: {precision_dm:.4f}")
print(f"  F1: {f1_dm:.4f} | Accuracy: {acc_dm:.4f}")

# Baseline 2: Estratégia estratificada
dummy_stratified = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DummyClassifier(strategy='stratified', random_state=42))
])

print("\nTreinando Dummy (stratified)...")
dummy_stratified.fit(X_tr, y_tr)
y_pred_ds = dummy_stratified.predict(X_val)
y_proba_ds = dummy_stratified.predict_proba(X_val)[:, 1]

roc_ds = roc_auc_score(y_val, y_proba_ds)
pr_ds = average_precision_score(y_val, y_proba_ds)
recall_ds = recall_score(y_val, y_pred_ds)
precision_ds = precision_score(y_val, y_pred_ds, zero_division=0)
f1_ds = f1_score(y_val, y_pred_ds, zero_division=0)
acc_ds = accuracy_score(y_val, y_pred_ds)

print(f"\nDummy (stratified):")
print(f"  ROC-AUC: {roc_ds:.4f} | PR-AUC: {pr_ds:.4f}")
print(f"  Recall: {recall_ds:.4f} | Precision: {precision_ds:.4f}")
print(f"  F1: {f1_ds:.4f} | Accuracy: {acc_ds:.4f}")

# Armazenar resultados
all_results.append({
    'model_name': 'Dummy (most_frequent)',
    'roc_auc': roc_dm, 'pr_auc': pr_dm,
    'recall': recall_dm, 'precision': precision_dm,
    'f1': f1_dm, 'accuracy': acc_dm,
    'cv_mean': None, 'cv_std': None
})
trained_models['Dummy (most_frequent)'] = dummy_majority

all_results.append({
    'model_name': 'Dummy (stratified)',
    'roc_auc': roc_ds, 'pr_auc': pr_ds,
    'recall': recall_ds, 'precision': precision_ds,
    'f1': f1_ds, 'accuracy': acc_ds,
    'cv_mean': None, 'cv_std': None
})
trained_models['Dummy (stratified)'] = dummy_stratified

print("\n✅ Baseline concluído!")

In [0]:
# ============================================================
# SEÇÃO 5 — LOGISTIC REGRESSION
# ============================================================
print("=" * 60)
print("SEÇÃO 5 — Logistic Regression (class_weight='balanced')")
print("=" * 60)

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        class_weight='balanced',
        max_iter=2000,
        solver='lbfgs',
        random_state=42,
        n_jobs=-1
    ))
])

print("Treinando Logistic Regression...")
t0 = time.time()
lr_pipeline.fit(X_tr, y_tr)
train_time_lr = time.time() - t0
print(f"Tempo de treinamento: {train_time_lr:.1f}s")

# Avaliar no validation
y_pred_lr = lr_pipeline.predict(X_val)
y_proba_lr = lr_pipeline.predict_proba(X_val)[:, 1]

roc_lr = roc_auc_score(y_val, y_proba_lr)
pr_lr = average_precision_score(y_val, y_proba_lr)
recall_lr = recall_score(y_val, y_pred_lr)
precision_lr = precision_score(y_val, y_pred_lr, zero_division=0)
f1_lr = f1_score(y_val, y_pred_lr, zero_division=0)
acc_lr = accuracy_score(y_val, y_pred_lr)

print(f"\nLogistic Regression:")
print(f"  ROC-AUC: {roc_lr:.4f} | PR-AUC: {pr_lr:.4f}")
print(f"  Recall: {recall_lr:.4f} | Precision: {precision_lr:.4f}")
print(f"  F1: {f1_lr:.4f} | Accuracy: {acc_lr:.4f}")

# Armazenar
all_results.append({
    'model_name': 'Logistic Regression',
    'roc_auc': roc_lr, 'pr_auc': pr_lr,
    'recall': recall_lr, 'precision': precision_lr,
    'f1': f1_lr, 'accuracy': acc_lr,
    'cv_mean': None, 'cv_std': None,
    'train_time': train_time_lr
})
trained_models['Logistic Regression'] = lr_pipeline

print("\n✅ Logistic Regression concluída!")

In [0]:
# ============================================================
# SEÇÃO 6 — RANDOM FOREST
# ============================================================
print("=" * 60)
print("SEÇÃO 6 — Random Forest (200 árvores, profundidade controlada)")
print("=" * 60)

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=50,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    ))
])

print("Treinando Random Forest...")
t0 = time.time()
rf_pipeline.fit(X_tr, y_tr)
train_time_rf = time.time() - t0
print(f"Tempo de treinamento: {train_time_rf:.1f}s")

# Avaliar
y_pred_rf = rf_pipeline.predict(X_val)
y_proba_rf = rf_pipeline.predict_proba(X_val)[:, 1]

roc_rf = roc_auc_score(y_val, y_proba_rf)
pr_rf = average_precision_score(y_val, y_proba_rf)
recall_rf = recall_score(y_val, y_pred_rf)
precision_rf = precision_score(y_val, y_pred_rf, zero_division=0)
f1_rf = f1_score(y_val, y_pred_rf, zero_division=0)
acc_rf = accuracy_score(y_val, y_pred_rf)

print(f"\nRandom Forest:")
print(f"  ROC-AUC: {roc_rf:.4f} | PR-AUC: {pr_rf:.4f}")
print(f"  Recall: {recall_rf:.4f} | Precision: {precision_rf:.4f}")
print(f"  F1: {f1_rf:.4f} | Accuracy: {acc_rf:.4f}")

# Armazenar
all_results.append({
    'model_name': 'Random Forest',
    'roc_auc': roc_rf, 'pr_auc': pr_rf,
    'recall': recall_rf, 'precision': precision_rf,
    'f1': f1_rf, 'accuracy': acc_rf,
    'cv_mean': None, 'cv_std': None,
    'train_time': train_time_rf
})
trained_models['Random Forest'] = rf_pipeline

print("\n✅ Random Forest concluído!")

In [0]:
# ============================================================
# SEÇÃO 7 — XGBOOST
# ============================================================
print("=" * 60)
print("SEÇÃO 7 — XGBoost")
print("=" * 60)

if xgb_available:
    xgb_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(
            scale_pos_weight=scale_pos_weight,
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            eval_metric='logloss'
        ))
    ])
    
    print("Treinando XGBoost...")
    t0 = time.time()
    xgb_pipeline.fit(X_tr, y_tr)
    train_time_xgb = time.time() - t0
    print(f"Tempo de treinamento: {train_time_xgb:.1f}s")
    
    # Avaliar
    y_pred_xgb = xgb_pipeline.predict(X_val)
    y_proba_xgb = xgb_pipeline.predict_proba(X_val)[:, 1]
    
    roc_xgb = roc_auc_score(y_val, y_proba_xgb)
    pr_xgb = average_precision_score(y_val, y_proba_xgb)
    recall_xgb = recall_score(y_val, y_pred_xgb)
    precision_xgb = precision_score(y_val, y_pred_xgb, zero_division=0)
    f1_xgb = f1_score(y_val, y_pred_xgb, zero_division=0)
    acc_xgb = accuracy_score(y_val, y_pred_xgb)
    
    print(f"\nXGBoost:")
    print(f"  ROC-AUC: {roc_xgb:.4f} | PR-AUC: {pr_xgb:.4f}")
    print(f"  Recall: {recall_xgb:.4f} | Precision: {precision_xgb:.4f}")
    print(f"  F1: {f1_xgb:.4f} | Accuracy: {acc_xgb:.4f}")
    
    all_results.append({
        'model_name': 'XGBoost',
        'roc_auc': roc_xgb, 'pr_auc': pr_xgb,
        'recall': recall_xgb, 'precision': precision_xgb,
        'f1': f1_xgb, 'accuracy': acc_xgb,
        'cv_mean': None, 'cv_std': None,
        'train_time': train_time_xgb
    })
    trained_models['XGBoost'] = xgb_pipeline
    print("\n✅ XGBoost concluído!")
else:
    print("⚠️ WARNING: XGBoost não disponível. Pulando este modelo.")
    print("Para usar XGBoost, instale com: %pip install xgboost")

In [0]:
# ============================================================
# SEÇÃO 8 — LIGHTGBM
# ============================================================
print("=" * 60)
print("SEÇÃO 8 — LightGBM")
print("=" * 60)

if lgb_available:
    lgb_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LGBMClassifier(
            scale_pos_weight=scale_pos_weight,
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        ))
    ])
    
    print("Treinando LightGBM...")
    t0 = time.time()
    lgb_pipeline.fit(X_tr, y_tr)
    train_time_lgb = time.time() - t0
    print(f"Tempo de treinamento: {train_time_lgb:.1f}s")
    
    # Avaliar
    y_pred_lgb = lgb_pipeline.predict(X_val)
    y_proba_lgb = lgb_pipeline.predict_proba(X_val)[:, 1]
    
    roc_lgb = roc_auc_score(y_val, y_proba_lgb)
    pr_lgb = average_precision_score(y_val, y_proba_lgb)
    recall_lgb = recall_score(y_val, y_pred_lgb)
    precision_lgb = precision_score(y_val, y_pred_lgb, zero_division=0)
    f1_lgb = f1_score(y_val, y_pred_lgb, zero_division=0)
    acc_lgb = accuracy_score(y_val, y_pred_lgb)
    
    print(f"\nLightGBM:")
    print(f"  ROC-AUC: {roc_lgb:.4f} | PR-AUC: {pr_lgb:.4f}")
    print(f"  Recall: {recall_lgb:.4f} | Precision: {precision_lgb:.4f}")
    print(f"  F1: {f1_lgb:.4f} | Accuracy: {acc_lgb:.4f}")
    
    all_results.append({
        'model_name': 'LightGBM',
        'roc_auc': roc_lgb, 'pr_auc': pr_lgb,
        'recall': recall_lgb, 'precision': precision_lgb,
        'f1': f1_lgb, 'accuracy': acc_lgb,
        'cv_mean': None, 'cv_std': None,
        'train_time': train_time_lgb
    })
    trained_models['LightGBM'] = lgb_pipeline
    print("\n✅ LightGBM concluído!")
else:
    print("⚠️ WARNING: LightGBM não disponível. Pulando este modelo.")
    print("Para usar LightGBM, instale com: %pip install lightgbm")

In [0]:
# ============================================================
# SEÇÃO 9 — CROSS VALIDATION (STRATIFIED K-FOLD, K=5)
# ============================================================
print("=" * 60)
print("SEÇÃO 9 — Cross Validation (Stratified KFold, k=5)")
print("=" * 60)

# Otimização: pré-transformar dados uma vez (evita refitar preprocessor por fold)
print("Pré-processando dados para CV (uma vez)...")
t0_prep = time.time()
X_transformed = preprocessor.fit_transform(X)
print(f"Pré-processamento concluído: {X_transformed.shape} ({time.time()-t0_prep:.1f}s)")

# Subamostragem para CV (limitação de tempo — 307K registros é muito lento para 5 folds)
cv_sample_size = 100000
if X_transformed.shape[0] > cv_sample_size:
    X_cv, _, y_cv, _ = train_test_split(
        X_transformed, y,
        train_size=cv_sample_size,
        stratify=y,
        random_state=42
    )
    print(f"Subamostra para CV: {cv_sample_size:,} registros (otimização de tempo)")
else:
    X_cv = X_transformed
    y_cv = y
    print(f"Usando dataset completo para CV: {X_cv.shape[0]:,} registros")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Criar novos classificadores para CV (mesmos hiperparâmetros)
cv_classifiers = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=2000, solver='lbfgs', random_state=42, n_jobs=-1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_leaf=50,
        class_weight='balanced', n_jobs=-1, random_state=42
    )
}
if xgb_available:
    cv_classifiers['XGBoost'] = XGBClassifier(
        scale_pos_weight=scale_pos_weight, n_estimators=200, max_depth=6,
        learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, eval_metric='logloss'
    )
if lgb_available:
    cv_classifiers['LightGBM'] = LGBMClassifier(
        scale_pos_weight=scale_pos_weight, n_estimators=200, max_depth=6,
        learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbose=-1
    )

print(f"\nModelos para CV: {list(cv_classifiers.keys())}")
print(f"Folds: 5 (Stratified)\n")

for model_name, clf in cv_classifiers.items():
    print(f"--- CV: {model_name} ---")
    t0 = time.time()
    scores = cross_val_score(
        clf, X_cv, y_cv,
        cv=skf,
        scoring='roc_auc',
        n_jobs=1  # Sequencial entre folds; paralelo dentro do modelo
    )
    cv_time = time.time() - t0
    cv_mean = scores.mean()
    cv_std = scores.std()
    
    cv_results[model_name] = {'mean': cv_mean, 'std': cv_std, 'scores': scores.tolist()}
    
    print(f"  ROC-AUC por fold: {[f'{s:.4f}' for s in scores]}")
    print(f"  ROC-AUC médio: {cv_mean:.4f} ± {cv_std:.4f}")
    print(f"  Tempo: {cv_time:.1f}s\n")

# Atualizar all_results com resultados de CV
for result in all_results:
    name = result['model_name']
    if name in cv_results:
        result['cv_mean'] = cv_results[name]['mean']
        result['cv_std'] = cv_results[name]['std']

print("✅ Cross Validation concluída!")

In [0]:
# ============================================================
# SEÇÃO 10 — THRESHOLD ANALYSIS
# ============================================================
print("=" * 60)
print("SEÇÃO 10 — Threshold Analysis")
print("=" * 60)

# Avaliar múltiplos thresholds para os melhores modelos (excluir Dummy)
non_dummy_models = {name: model for name, model in trained_models.items()
                    if not name.startswith('Dummy')}

thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]
threshold_results = []

print(f"Thresholds avaliados: {thresholds}\n")

for model_name, model_pipeline in non_dummy_models.items():
    print(f"--- {model_name} ---")
    y_proba = model_pipeline.predict_proba(X_val)[:, 1]
    
    for thresh in thresholds:
        y_pred_thresh = (y_proba >= thresh).astype(int)
        prec = precision_score(y_val, y_pred_thresh, zero_division=0)
        rec = recall_score(y_val, y_pred_thresh)
        f1_t = f1_score(y_val, y_pred_thresh, zero_division=0)
        
        threshold_results.append({
            'model_name': model_name,
            'threshold': thresh,
            'precision': prec,
            'recall': rec,
            'f1': f1_t
        })
        print(f"  Threshold {thresh:.2f}: Precision={prec:.4f} | Recall={rec:.4f} | F1={f1_t:.4f}")
    print()

threshold_df = pd.DataFrame(threshold_results)
print("✅ Threshold Analysis concluída! Trade-offs identificados.")
print("Nota: Nenhum modelo foi alterado — apenas análise de thresholds.")

In [0]:
# ============================================================
# SEÇÃO 11 — FEATURE IMPORTANCE (TOP 30)
# ============================================================
print("=" * 60)
print("SEÇÃO 11 — Feature Importance (Top 30)")
print("=" * 60)

# Modelos que suportam feature_importances_
importance_models = ['Random Forest', 'XGBoost', 'LightGBM']

for model_name in importance_models:
    if model_name not in trained_models:
        print(f"\n⚠️ {model_name} não disponível — pulando feature importance")
        continue
    
    print(f"\n--- {model_name} ---")
    model_pipeline = trained_models[model_name]
    classifier = model_pipeline.named_steps['classifier']
    
    # Obter importâncias
    importances = classifier.feature_importances_
    
    # Obter nomes das features após preprocessor
    try:
        feature_names = model_pipeline.named_steps['preprocessor'].get_feature_names_out()
        # Limpar prefixos do ColumnTransformer (num__, cat__)
        feature_names = [name.replace('num__', '').replace('cat__', '') for name in feature_names]
    except Exception:
        feature_names = [f'feature_{i}' for i in range(len(importances))]
    
    # Criar DataFrame com ranking
    fi_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False).reset_index(drop=True)
    fi_df['ranking'] = fi_df.index + 1
    feature_importances[model_name] = fi_df
    
    print("Top 30 features:")
    print(fi_df.head(30).to_string(index=False))

# Comparar top features entre modelos
print("\n--- Comparação de Top Features entre Modelos ---")
available_importance = {k: v for k, v in feature_importances.items() if len(v) > 0}
if len(available_importance) >= 2:
    top_features_comparison = pd.DataFrame()
    for name, fi_df in available_importance.items():
        top_30 = fi_df.head(30)[['feature', 'importance']].rename(
            columns={'importance': f'imp_{name.replace(" ", "_")}'}
        )
        if top_features_comparison.empty:
            top_features_comparison = top_30
        else:
            top_features_comparison = top_features_comparison.merge(
                top_30, on='feature', how='outer'
            )
    print(top_features_comparison.head(30).to_string(index=False))
else:
    print("Apenas um modelo com feature importance disponível.")

print("\n✅ Feature Importance concluída!")

In [0]:
# ============================================================
# SEÇÃO 12 — MATRIZ DE CONFUSÃO
# ============================================================
print("=" * 60)
print("SEÇÃO 12 — Matriz de Confusão")
print("=" * 60)

# Avaliar todos os modelos (incluindo Dummy para referência)
confusion_results = []

for model_name, model_pipeline in trained_models.items():
    print(f"\n--- {model_name} ---")
    y_pred = model_pipeline.predict(X_val)
    cm = confusion_matrix(y_val, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    print(f"  TN={tn:,} | FP={fp:,} | FN={fn:,} | TP={tp:,}")
    print(f"  Matriz:")
    print(f"  [{tn:>8,}  {fp:>8,}]")
    print(f"  [{fn:>8,}  {tp:>8,}]")
    
    confusion_results.append({
        'model_name': model_name,
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp
    })

confusion_df = pd.DataFrame(confusion_results)
print(f"\n✅ Matriz de Confusão concluída para {len(confusion_df)} modelos!")

In [0]:
# ============================================================
# SEÇÃO 13 — CURVAS ROC E PRECISION-RECALL
# ============================================================
print("=" * 60)
print("SEÇÃO 13 — Curvas ROC e Precision-Recall")
print("=" * 60)

# Modelos para plotar (excluir Dummy — sem valor preditivo)
plot_models = {name: model for name, model in trained_models.items()
               if not name.startswith('Dummy')}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- ROC Curve ---
ax_roc = axes[0]
for model_name, model_pipeline in plot_models.items():
    y_proba = model_pipeline.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_proba)
    roc_auc_val = roc_auc_score(y_val, y_proba)
    ax_roc.plot(fpr, tpr, label=f'{model_name} (AUC={roc_auc_val:.4f})')

ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random (AUC=0.5)')
ax_roc.set_xlabel('False Positive Rate')
ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title('ROC Curve — Comparação de Modelos')
ax_roc.legend(loc='lower right', fontsize=9)
ax_roc.grid(True, alpha=0.3)

# --- Precision-Recall Curve ---
ax_pr = axes[1]
for model_name, model_pipeline in plot_models.items():
    y_proba = model_pipeline.predict_proba(X_val)[:, 1]
    precision_arr, recall_arr, _ = precision_recall_curve(y_val, y_proba)
    pr_auc_val = average_precision_score(y_val, y_proba)
    ax_pr.plot(recall_arr, precision_arr, label=f'{model_name} (AP={pr_auc_val:.4f})')

# Linha de referência (proporção da classe positiva)
baseline_pr = y_val.mean()
ax_pr.axhline(y=baseline_pr, color='k', linestyle='--', alpha=0.3, label=f'Baseline ({baseline_pr:.4f})')
ax_pr.set_xlabel('Recall')
ax_pr.set_ylabel('Precision')
ax_pr.set_title('Precision-Recall Curve — Comparação de Modelos')
ax_pr.legend(loc='upper right', fontsize=9)
ax_pr.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Curvas ROC e Precision-Recall geradas para todos os modelos!")

In [0]:
# ============================================================
# SEÇÃO 14 — COMPARAÇÃO FINAL
# ============================================================
print("=" * 60)
print("SEÇÃO 14 — Comparação Final")
print("=" * 60)

# Criar tabela consolidada
comparison_df = pd.DataFrame(all_results)

# Reordenar colunas
display_cols = ['model_name', 'roc_auc', 'pr_auc', 'recall', 'precision', 'f1', 'accuracy', 'cv_mean', 'cv_std']
comparison_df = comparison_df[display_cols].copy()

# Ordenar por ROC-AUC (desc) e depois PR-AUC (desc)
comparison_df = comparison_df.sort_values(['roc_auc', 'pr_auc'], ascending=[False, False]).reset_index(drop=True)

# Arredondar valores
for col in ['roc_auc', 'pr_auc', 'recall', 'precision', 'f1', 'accuracy']:
    comparison_df[col] = comparison_df[col].round(4)
if 'cv_mean' in comparison_df.columns:
    comparison_df['cv_mean'] = comparison_df['cv_mean'].round(4)
if 'cv_std' in comparison_df.columns:
    comparison_df['cv_std'] = comparison_df['cv_std'].round(4)

print("\n=== Tabela Consolidada de Resultados ===")
print()
print(comparison_df.to_string(index=False))

print(f"\n\nCritério de ordenação: ROC-AUC (primário), PR-AUC (secundário)")
print("NOTA: Accuracy NÃO é utilizada como critério principal (dataset desbalanceado).")
print("\n✅ Comparação final concluída!")

In [0]:
# ============================================================
# SEÇÃO 15 — SELEÇÃO DE CAMPEÕES
# ============================================================
print("=" * 60)
print("SEÇÃO 15 — Seleção de Campeões")
print("=" * 60)

# Filtrar apenas modelos reais (excluir Dummy)
real_results = [r for r in all_results if not r['model_name'].startswith('Dummy')]
real_df = pd.DataFrame(real_results)

# Campeões por categoria
champions = {}

# Melhor ROC-AUC
best_roc = real_df.loc[real_df['roc_auc'].idxmax()]
champions['ROC-AUC'] = best_roc['model_name']
print(f"🏆 Melhor ROC-AUC: {best_roc['model_name']} ({best_roc['roc_auc']:.4f})")

# Melhor PR-AUC
best_pr = real_df.loc[real_df['pr_auc'].idxmax()]
champions['PR-AUC'] = best_pr['model_name']
print(f"🏆 Melhor PR-AUC: {best_pr['model_name']} ({best_pr['pr_auc']:.4f})")

# Melhor Recall
best_recall = real_df.loc[real_df['recall'].idxmax()]
champions['Recall'] = best_recall['model_name']
print(f"🏆 Melhor Recall: {best_recall['model_name']} ({best_recall['recall']:.4f})")

# Melhor F1
best_f1 = real_df.loc[real_df['f1'].idxmax()]
champions['F1'] = best_f1['model_name']
print(f"🏆 Melhor F1: {best_f1['model_name']} ({best_f1['f1']:.4f})")

print(f"\n--- Resumo de Campeões ---")
for category, model in champions.items():
    print(f"  {category}: {model}")

# Verificar se um modelo ganhou múltiplas categorias
from collections import Counter
champion_counts = Counter(champions.values())
multi_champions = {model: count for model, count in champion_counts.items() if count > 1}

if multi_champions:
    print(f"\n🌟 Modelo(s) com múltiplas vitórias:")
    for model, count in multi_champions.items():
        cats = [k for k, v in champions.items() if v == model]
        print(f"  {model}: {count} categorias ({', '.join(cats)})")

print("\nNOTA: Um mesmo modelo pode vencer várias categorias.")
print("Accuracy NÃO é utilizada como critério de seleção (dataset desbalanceado).")
print("\n✅ Seleção de campeões concluída!")

In [0]:
# ============================================================
# SEÇÃO 16 — TABELAS DE RESULTADO
# ============================================================
print("=" * 60)
print("SEÇÃO 16 — Tabelas de Resultado")
print("=" * 60)

# Criar schema analytics se não existir
spark.sql("CREATE SCHEMA IF NOT EXISTS credit_risk.analytics")
print("✅ Schema credit_risk.analytics garantido")

# --- Tabela 1: model_training_results ---
print("\n--- Criando credit_risk.analytics.model_training_results ---")

# Preparar DataFrame de resultados
results_for_table = pd.DataFrame(all_results)
results_for_table['execution_id'] = EXECUTION_ID
results_for_table['train_timestamp'] = pd.Timestamp(START_TIMESTAMP)

# Selecionar e reordenar colunas
results_cols = ['execution_id', 'model_name', 'roc_auc', 'pr_auc', 'recall', 'precision', 
                'f1', 'accuracy', 'cv_mean', 'cv_std', 'train_timestamp']
results_for_table = results_for_table[results_cols]

# Converter para Spark e salvar como tabela Delta (append)
results_spark_df = spark.createDataFrame(results_for_table)
results_spark_df.write.mode("append").format("delta").saveAsTable("credit_risk.analytics.model_training_results")
print(f"✅ model_training_results: {len(results_for_table)} registros inseridos (append)")

# --- Tabela 2: feature_importance_training ---
print("\n--- Criando credit_risk.analytics.feature_importance_training ---")
fi_all = []
for model_name, fi_df in feature_importances.items():
    fi_copy = fi_df.copy()
    fi_copy['model_name'] = model_name
    fi_all.append(fi_copy)

if fi_all:
    fi_combined = pd.concat(fi_all, ignore_index=True)
    fi_combined = fi_combined[['model_name', 'feature', 'importance', 'ranking']]
    
    fi_spark_df = spark.createDataFrame(fi_combined)
    fi_spark_df.write.mode("append").format("delta").saveAsTable("credit_risk.analytics.feature_importance_training")
    print(f"✅ feature_importance_training: {len(fi_combined)} registros inseridos (append)")
else:
    print("⚠️ Nenhuma feature importance disponível para salvar")

print("\n✅ Tabelas de resultado criadas com sucesso!")

In [0]:
# ============================================================
# SEÇÃO 17 — AUDITORIA
# ============================================================
print("=" * 60)
print("SEÇÃO 17 — Auditoria")
print("=" * 60)

END_TIME = time.time()
END_TIMESTAMP = datetime.now()
DURATION = END_TIME - START_TIME

# Identificar melhor modelo e métricas
real_results_audit = [r for r in all_results if not r['model_name'].startswith('Dummy')]
best_model_row = max(real_results_audit, key=lambda x: x['roc_auc'])
best_model = best_model_row['model_name']
best_roc_auc = best_model_row['roc_auc']
best_pr_auc = max(r['pr_auc'] for r in real_results_audit)

# Contar modelos treinados (excluir Dummy)
n_models_trained = len(real_results_audit)

# Determinar status
status = "SUCCESS"
if n_models_trained < 2:
    status = "WARNING"

# Criar registro de auditoria
audit_record = pd.DataFrame([{
    'execution_id': EXECUTION_ID,
    'start_time': START_TIMESTAMP,
    'end_time': END_TIMESTAMP,
    'duration_seconds': round(DURATION, 2),
    'models_trained': n_models_trained,
    'best_model': best_model,
    'best_roc_auc': round(best_roc_auc, 4),
    'best_pr_auc': round(best_pr_auc, 4),
    'n_features': len(feature_cols),
    'n_rows_train': len(X),
    'status': status
}])

print("\n--- Registro de Auditoria ---")
print(audit_record.to_string(index=False))

# Salvar como tabela Delta append-only
audit_spark_df = spark.createDataFrame(audit_record)
audit_spark_df.write.mode("append").format("delta").saveAsTable("credit_risk.analytics.audit_training")
print(f"\n✅ credit_risk.analytics.audit_training: registro inserido (append-only)")
print(f"\nDuração total: {DURATION:.1f}s ({DURATION/60:.1f} min)")

In [0]:
# ============================================================
# SEÇÃO 18 — RESUMO EXECUTIVO
# ============================================================
print("=" * 60)
print("TRAINING RESULTS")
print("=" * 60)

print(f"\nModelos treinados: {n_models_trained}")

print(f"\nMelhor ROC-AUC:")
print(f"  {best_roc['model_name']} — {best_roc['roc_auc']:.4f}")

print(f"\nMelhor PR-AUC:")
print(f"  {best_pr['model_name']} — {best_pr['pr_auc']:.4f}")

print(f"\nMelhor Recall:")
print(f"  {best_recall['model_name']} — {best_recall['recall']:.4f}")

print(f"\nMelhor F1:")
print(f"  {best_f1['model_name']} — {best_f1['f1']:.4f}")

print(f"\nDataset:")
print(f"  {len(X):,} registros")
print(f"  {len(feature_cols)} features")

print(f"\nStatus:")
print(f"  {status}")

print(f"\nTabelas criadas:")
print(f"  credit_risk.analytics.model_training_results")
print(f"  credit_risk.analytics.feature_importance_training")
print(f"  credit_risk.analytics.audit_training")

print(f"\nExecution ID: {EXECUTION_ID}")
print(f"Duração: {DURATION:.1f}s ({DURATION/60:.1f} min)")
print(f"\n{'=' * 60}")